# 🚀 Advanced PySpark Performance Tuning & Optimization (Part 4)
This notebook completes our masterclass series by diving into production storage layer optimizations, streaming state store performance, and vectorized Arrow operations: Delta Lake ACID File Compaction, Hive Bucketing vs. Partitioning, RocksDB Stateful Streaming, and Vectorized Arrow Interoperability.

--- 
## 13. Delta Lake Optimization (OPTIMIZE & Z-ORDER)
### **Concept & Architectural Explanation**
* **The Small-File Bottleneck:** Streaming ingests or frequent micro-batches generate thousands of small Parquet files in object storage (S3/ADLS/GCS), creating severe metadata listing overhead.
* **How OPTIMIZE Compaction Works:** Delta Lake consolidates small files into uniform ~1GB files.
* **How Z-Ordering Works:** Z-Ordering clusters data along multi-dimensional space-filling curves. Unlike standard single-column partitioning, Z-Ordering allows predicate pushdown pruning on multiple high-cardinality search keys simultaneously.
* **Production Best Practice:** Run `OPTIMIZE table ZORDER BY (col1, col2)` on a scheduled maintenance job after heavy data writes.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import os
import shutil

# Initialize SparkSession with Delta Lake Extensions
spark = SparkSession.builder \
    .appName("Optimization_13_Delta_Lake") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .master("local[*]") \
    .getOrCreate()

delta_path = "/tmp/delta_optimization_demo"
if os.path.exists(delta_path):
    shutil.rmtree(delta_path)

# Step 1: Write multiple small micro-batch appends to simulate file fragmentation
print("=== Simulating Micro-batch Writes (Small File Issue) ===")
for batch_id in range(5):
    df_batch = spark.range(batch_id * 1000, (batch_id + 1) * 1000) \
        .withColumn("user_id", (F.col("id") % 100).cast("int")) \
        .withColumn("region", F.when(F.col("id") % 2 == 0, "US").otherwise("EU"))
    df_batch.write.format("delta").mode("append").save(delta_path)

# Step 2: Query table before optimization and check physical plan
df_delta = spark.read.format("delta").load(delta_path)
filtered_delta = df_delta.filter("user_id = 42 AND region = 'US'")
print("=== Physical Plan: Pre-Optimization Delta Read ===")
filtered_delta.explain(True)

print("\nDelta Lake Compaction Syntax: OPTIMIZE delta_table ZORDER BY (user_id, region)")

--- 
## 14. Hive Bucketing vs. Partitioning for Zero-Shuffle Joins
### **Concept & Architectural Explanation**
* **The Shuffle Problem:** Joining two massive tables typically requires an `Exchange hashpartitioning` across the network to align join keys on the same node.
* **How Bucketing Works:** Pre-sorts and pre-hashes dataset rows into fixed $N$ bucket files saved on disk using `.bucketBy(num_buckets, "join_key").sortBy("join_key")` inside Hive metastore tables.
* **The Benefit:** When joining two bucketed tables with identical bucket counts on the join key, Spark completely skips both the `Exchange` (shuffle) phase AND the `Sort` phase, executing a zero-shuffle join directly from disk.
* **Production Best Practice:** Use bucketing for high-cardinality join keys (e.g., `user_id`, `device_id`) where directory partitioning would create millions of tiny folders.

In [ ]:
print("=== Concept Demonstration: Bucketing Configuration ===")
# Example setup code for saving bucketed tables (requires metastore catalog enabled)
sample_data = spark.range(0, 100000).withColumn("join_key", F.col("id") % 50)

# Bucketing syntax for metastore tables:
# sample_data.write \
#     .format("parquet") \
#     .bucketBy(16, "join_key") \
#     .sortBy("join_key") \
#     .saveAsTable("bucketed_transactions")

print("Bucketing configuration parameters established for Hive catalog persistence.")

--- 
## 15. Stateful Structured Streaming Optimization (RocksDB State Store)
### **Concept & Architectural Explanation**
* **The Default HDFS State Store Issue:** The default in-memory Java heap state store maintains stateful streaming checkpoints (e.g., aggregation windows, stream-stream joins) inside the JVM. Large state datasets cause long JVM Garbage Collection pauses and executor OOM crashes.
* **How RocksDB Provider Fixes It:** Replaces JVM heap state with **RocksDB**, an embedded high-performance key-value store using off-heap native memory and local SSD disk storage.
* **Production Best Practice:** Always enable RocksDB state store for stateful structured streaming applications managing long-running state windows.

In [ ]:
# Step 1: Enable RocksDB State Store Provider Configuration
spark.conf.set(
    "spark.sql.streaming.stateStore.providerClass",
    "org.apache.spark.sql.execution.streaming.state.RocksDBStateStoreProvider"
)

print("Active State Store Provider:", spark.conf.get("spark.sql.streaming.stateStore.providerClass"))
print("RocksDB configured successfully for low-pause off-heap state management.")

--- 
## 16. PyArrow Serialization Optimization (`spark.sql.execution.arrow.pyspark.enabled`)
### **Concept & Architectural Explanation**
* **The Serialization Bottleneck:** Converting PySpark DataFrames to native Python structures like Pandas via `.toPandas()` or `spark.createDataFrame(pandas_df)` uses standard Python pickle serialization, which is painfully slow for large datasets.
* **How Apache Arrow Optimizes:** Uses zero-copy columnar in-memory data representations to exchange data between PySpark and Pandas/Python without CPU-intensive serialization loops.
* **Production Best Practice:** Ensure `spark.sql.execution.arrow.pyspark.enabled = true` is set when performing ML feature engineering or bridging PySpark data into Python data science libraries.

In [ ]:
import time

# Step 1: Enable PyArrow Vectorization
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

# Step 2: Create large PySpark DataFrame and measure Apache Arrow export speed
df_large = spark.range(0, 1000000).withColumn("value", F.rand())

start = time.time()
pandas_df = df_large.toPandas()
duration = time.time() - start

print(f"Exported 1,000,000 rows to Pandas via Apache Arrow in: {duration:.2f} seconds")
print(f"Pandas Shape: {pandas_df.shape}")

spark.stop()